# Atelier Préparation de Données Images

## Contexte

Une entreprise souhaite développer un système d'intelligence artificielle capable de reconnaître automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des déchets. Le modèle devra classer chaque image dans l'une des catégories suivantes :

- **cardboard** : cartons ondulés, cartons plats...
- **plastic** : bouteilles, emballages plastiques...
- **paper** : feuilles, journaux...
- **glass** : bouteilles et objets en verre...
- **metal** : canettes, boîtes métalliques...
- **trash** : emballages bonbons, tasses jetables...

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas homogènes : dimensions différentes, formats différents, images RGB et grayscale, certaines images sont trop petites, certaines images sont corrompues, quelques images sont vides, images dupliquées, quelques images placées dans le mauvais dossier, classes déséquilibrées.

L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning.

# Partie 1 – Exploration du dataset

In [1]:
# Partie 1 - Étape 1 : Imports nécessaires
import os
import numpy as np
import pandas as pd
from PIL import Image

In [2]:
# Partie 1 - Étape 2 : Fonction d'extraction des métadonnées d'une image
def extraire_metadonnees(chemin_image, classe):
    """
    Extrait les métadonnées d'une image : nom, classe, format, mode,
    largeur, hauteur, écart-type des pixels, nombre de canaux, taille.
    Retourne un dictionnaire, avec corrompue=True si l'image ne peut pas être lue.
    """
    nom_fichier = os.path.basename(chemin_image)
    taille_octets = os.path.getsize(chemin_image)

    try:
        with Image.open(chemin_image) as img:
            img.verify()  # vérifie l'intégrité du fichier sans le charger complètement

        # Réouverture nécessaire après verify() (qui "consomme" l'objet image)
        with Image.open(chemin_image) as img:
            largeur, hauteur = img.size
            format_img = img.format
            mode_img = img.mode

            img_array = np.array(img)
            ecart_type = img_array.std()
            nb_canaux = 1 if img_array.ndim == 2 else img_array.shape[2]

        return {
            'nom_fichier': nom_fichier,
            'classe': classe,
            'format': format_img,
            'mode': mode_img,
            'largeur': largeur,
            'hauteur': hauteur,
            'ecart_type_pixels': ecart_type,
            'nb_canaux': nb_canaux,
            'taille_octets': taille_octets,
            'corrompue': False
        }

    except Exception as e:
        return {
            'nom_fichier': nom_fichier,
            'classe': classe,
            'format': None,
            'mode': None,
            'largeur': None,
            'hauteur': None,
            'ecart_type_pixels': None,
            'nb_canaux': None,
            'taille_octets': taille_octets,
            'corrompue': True
        }

In [3]:
# Partie 1 - Étape 3 : Parcours de tout le dataset et construction du tableau récapitulatif
dossier_raw = "../data/raw"
classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

resultats = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in os.listdir(dossier_classe):
        chemin_complet = os.path.join(dossier_classe, nom_fichier)
        metadonnees = extraire_metadonnees(chemin_complet, classe)
        resultats.append(metadonnees)

df_images = pd.DataFrame(resultats)
df_images.shape

(1032, 10)

In [4]:
# Partie 1 - Étape 4 : Aperçu du résultat
print(df_images.head())
print("\nNombre d'images par classe :")
print(df_images['classe'].value_counts())
print("\nNombre d'images corrompues :", df_images['corrompue'].sum())

        nom_fichier     classe format mode  largeur  hauteur  \
0    cardboard1.jpg  cardboard   JPEG  RGB    512.0    384.0   
1   cardboard10.jpg  cardboard   JPEG  RGB    512.0    384.0   
2  cardboard100.jpg  cardboard   JPEG  RGB    512.0    384.0   
3  cardboard101.jpg  cardboard   JPEG  RGB    512.0    384.0   
4  cardboard102.jpg  cardboard   JPEG  RGB    512.0    384.0   

   ecart_type_pixels  nb_canaux  taille_octets  corrompue  
0          40.586504        3.0          17333      False  
1          42.577273        3.0          21683      False  
2          46.121684        3.0          14884      False  
3          72.264255        3.0          14289      False  
4          48.389753        3.0          18015      False  

Nombre d'images par classe :
classe
paper        252
plastic      224
glass        188
cardboard    169
metal        149
trash         50
Name: count, dtype: int64

Nombre d'images corrompues : 6


# Partie 2 – Détecter les images corrompues

In [5]:
# Partie 2 : Fonction de détection d'image corrompue
def est_corrompue(chemin_image):
    """
    Vérifie si une image est corrompue (fichier illisible, tronqué,
    ou n'étant pas une image valide). Retourne True si corrompue, False sinon.
    """
    try:
        with Image.open(chemin_image) as img:
            img.verify()
        return False
    except Exception:
        return True

In [6]:
# Partie 2 : Application de la fonction sur tout le dataset
images_corrompues = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in os.listdir(dossier_classe):
        chemin_complet = os.path.join(dossier_classe, nom_fichier)
        if est_corrompue(chemin_complet):
            images_corrompues.append({'nom_fichier': nom_fichier, 'classe': classe})

df_corrompues = pd.DataFrame(images_corrompues)
print("Nombre d'images corrompues détectées :", len(df_corrompues))
df_corrompues

Nombre d'images corrompues détectées : 6


,nom_fichier,classe
0,cardboard83.jpg,cardboard
1,glass74.jpg,glass
2,metal48.jpg,metal
3,paper213.jpg,paper
4,plastic13.jpg,plastic
5,trash3.jpg,trash
